In [1]:
# 📚 Essential Imports and Setup

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment
import warnings
warnings.filterwarnings('ignore')

# 📁 Snapshot paths
SNAP_PATH = Path("C:/Users/Marco.Africani/OneDrive - AVU SA/AVU CPI Campaign/Puzzle_control_Reports/IRON_DATA/snapshots")

print("📚 Imports loaded successfully")
print(f"📁 Snapshot path: {SNAP_PATH}")
print(f"📊 Ready for analysis!")

📚 Imports loaded successfully
📁 Snapshot path: C:\Users\Marco.Africani\OneDrive - AVU SA\AVU CPI Campaign\Puzzle_control_Reports\IRON_DATA\snapshots
📊 Ready for analysis!


In [20]:
# 🧊 SNAPSHOT DATA LOADING - Reliable

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from pathlib import Path

print("🧊 SNAPSHOT DATA LOADING")
print("=" * 60)

# Setup paths and runtime
SNAP_PATH = Path(r"C:\Users\Marco.Africani\OneDrive - AVU SA\AVU CPI Campaign\Puzzle_control_Reports\IRON_DATA\snapshots")
analysis_runtime = datetime.now()

print(f"📅 Analysis Runtime: {analysis_runtime.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"📁 Snapshot Directory: {SNAP_PATH}")

# Time windows for SPEC compliance
time_windows = {
    '7d': analysis_runtime - timedelta(days=7),
    '14d': analysis_runtime - timedelta(days=14),
    '30d': analysis_runtime - timedelta(days=30)
}

print(f"\n⏰ Analysis Time Windows:")
for window, cutoff in time_windows.items():
    print(f"   {window}: Campaigns after {cutoff.strftime('%Y-%m-%d %H:%M')}")

# Load snapshot files
snapshot_files = {
    'omt_main_offer.pkl': 'OMT Main Offer',
    'powerbi_stats.pkl': 'Power BI Statistics', 
    'detailed_stock_list.pkl': 'Stock List',
    'clients_lines.pkl': 'Client Lines',
    'inactive2025.pkl': 'Inactive Clients'
}

loaded_data = {}
print(f"\n📊 Loading Snapshot Files:")

for filename, description in snapshot_files.items():
    try:
        snapshot_file = SNAP_PATH / filename
        if snapshot_file.exists():
            df = pd.read_pickle(snapshot_file)
            loaded_data[description] = df
            print(f"   ✅ {description}: {len(df):,} records × {len(df.columns)} columns")
        else:
            print(f"   ❌ {description}: Snapshot file not found")
            loaded_data[description] = None
    except Exception as e:
        print(f"   ❌ {description}: Error loading - {str(e)[:50]}...")
        loaded_data[description] = None

# Assign to global variables with proper names
omt_df = loaded_data.get('OMT Main Offer')
stats_df = loaded_data.get('Power BI Statistics')
stock_df = loaded_data.get('Stock List')
lines_df = loaded_data.get('Client Lines')
inactive_df = loaded_data.get('Inactive Clients')

# Quick data validation and column identification
print(f"\n🔍 Data Validation & Column Mapping:")

# OMT Main Offer validation
if omt_df is not None:
    print(f"   📋 OMT Main Offer ({len(omt_df)} campaigns):")
    schedule_cols = [col for col in omt_df.columns if 'schedule' in col.lower()]
    campaign_cols = [col for col in omt_df.columns if 'campaign' in col.lower()]
    email_cols = [col for col in omt_df.columns if 'email' in col.lower() or 'sent' in col.lower()]
    
    if schedule_cols:
        schedule_col = schedule_cols[0]
        print(f"      📅 Schedule column: '{schedule_col}'")
        try:
            omt_df['schedule_datetime'] = pd.to_datetime(omt_df[schedule_col], errors='coerce')
            valid_schedules = omt_df['schedule_datetime'].notna().sum()
            print(f"      ✅ Valid schedule dates: {valid_schedules}")
        except:
            print(f"      ⚠️  Schedule date parsing failed")
    
    if campaign_cols:
        campaign_col_omt = campaign_cols[0]
        print(f"      📋 Campaign column: '{campaign_col_omt}'")
    
    if email_cols:
        email_col = email_cols[0]
        print(f"      📧 Email column: '{email_col}'")

# Power BI Statistics validation (SPEC compliance)
if stats_df is not None:
    print(f"   📊 Power BI Statistics ({len(stats_df)} records):")
    
    # Find SPEC-required columns
    campaign_cols = [col for col in stats_df.columns if 'campaign' in col.lower() and 'no' in col.lower() and 'main' not in col.lower()]
    main_campaign_cols = [col for col in stats_df.columns if 'main' in col.lower() and 'campaign' in col.lower()]
    lcy_cols = [col for col in stats_df.columns if 'bottle' in col.lower() and 'lcy' in col.lower()]
    customer_cols = [col for col in stats_df.columns if 'customer' in col.lower()]
    segment_cols = [col for col in stats_df.columns if 'segment' in col.lower()]
    wine_cols = [col for col in stats_df.columns if 'wine' in col.lower()]
    size_cols = [col for col in stats_df.columns if 'bottle' in col.lower() and 'size' in col.lower()]
    
    # Store key columns globally
    campaign_col = campaign_cols[0] if campaign_cols else None
    main_campaign_col = main_campaign_cols[0] if main_campaign_cols else None
    lcy_col = lcy_cols[0] if lcy_cols else None
    customer_col = customer_cols[0] if customer_cols else None
    segment_col = segment_cols[0] if segment_cols else None
    wine_col = wine_cols[0] if wine_cols else None
    size_col = size_cols[0] if size_cols else None
    
    print(f"      📋 Campaign No: '{campaign_col}'")
    print(f"      📋 Main Campaign: '{main_campaign_col}'")
    print(f"      💰 LCY Amount: '{lcy_col}' {'✅ SPEC COMPLIANT' if lcy_col else '❌ NOT FOUND'}")
    print(f"      👤 Customer No: '{customer_col}'")
    print(f"      🏢 Segment: '{segment_col}'")
    print(f"      🍷 Wine Name: '{wine_col}'")
    print(f"      📏 Size: '{size_col}'")
    
    # SPEC: HORECA and Trade exclusion
    if segment_col:
        initial_count = len(stats_df)
        exclude_mask = stats_df[segment_col].str.contains('HORECA|Trade|Horeca|trade', case=False, na=False)
        stats_df = stats_df[~exclude_mask]
        excluded_count = initial_count - len(stats_df)
        print(f"      🚫 HORECA & Trade excluded: {excluded_count:,} records (remaining: {len(stats_df):,})")

# Client Lines validation
if lines_df is not None:
    print(f"   👥 Client Lines ({len(lines_df)} clients):")
    if len(lines_df.columns) >= 21:
        contact_col = lines_df.columns[20]  # Column U (index 20)
        print(f"      📞 Contact No (Col U): '{contact_col}'")
        unique_contacts = lines_df[contact_col].nunique()
        print(f"      👤 Unique contacts: {unique_contacts}")
    else:
        print(f"      ⚠️  Column U not available (only {len(lines_df.columns)} columns)")
    
    # Find spending column
    spending_cols = [col for col in lines_df.columns if 'spend' in col.lower() or 'avg' in col.lower()]
    if spending_cols:
        spending_col = spending_cols[0]
        print(f"      💰 Spending column: '{spending_col}'")

# Inactive clients validation
if inactive_df is not None:
    print(f"   😴 Inactive Clients ({len(inactive_df)} records):")
    contact_cols = [col for col in inactive_df.columns if 'contact' in col.lower()]
    if contact_cols:
        inactive_contact_col = contact_cols[0]
        print(f"      📞 Contact column: '{inactive_contact_col}'")

# Stock validation
if stock_df is not None:
    print(f"   📦 Stock List ({len(stock_df)} items):")
    item_cols = [col for col in stock_df.columns if 'item' in col.lower()]
    stock_cols = [col for col in stock_df.columns if 'stock' in col.lower() or 'qty' in col.lower()]
    if item_cols and stock_cols:
        item_col = item_cols[0]
        stock_col = stock_cols[0]
        print(f"      📋 Item column: '{item_col}'")
        print(f"      📊 Stock column: '{stock_col}'")

# Time window filtering for OMT campaigns (SPEC compliance)
if omt_df is not None and 'schedule_datetime' in omt_df.columns:
    print(f"\n⏰ OMT Campaign Filtering by Schedule DateTime:")
    omt_filtered = {}
    
    for window, cutoff_date in time_windows.items():
        window_campaigns = omt_df[omt_df['schedule_datetime'] >= cutoff_date]
        omt_filtered[window] = window_campaigns
        print(f"   {window}: {len(window_campaigns)} campaigns scheduled after {cutoff_date.strftime('%Y-%m-%d')}")
        
        # Show sample campaigns
        if len(window_campaigns) > 0:
            sample = window_campaigns.head(3)
            for idx, row in sample.iterrows():
                campaign_id = row.get(campaign_col_omt if 'campaign_col_omt' in locals() else 'Campaign No.', 'Unknown')
                schedule_date = row['schedule_datetime']
                print(f"      • {campaign_id}: {schedule_date.strftime('%Y-%m-%d %H:%M') if pd.notna(schedule_date) else 'Invalid date'}")
else:
    print(f"\n⚠️  Cannot perform time filtering - OMT schedule data unavailable")
    omt_filtered = {}

# Summary
print(f"\n" + "=" * 60)
print(f"📋 SNAPSHOT LOADING SUMMARY")
print(f"=" * 60)

loaded_count = sum(1 for df in loaded_data.values() if df is not None)
total_records = sum(len(df) for df in loaded_data.values() if df is not None)

print(f"✅ Files loaded: {loaded_count}/{len(snapshot_files)}")
print(f"📊 Total records: {total_records:,}")

for description, df in loaded_data.items():
    status = "✅ Ready" if df is not None else "❌ Missing"
    print(f"{status} {description}")

if omt_filtered:
    total_campaigns_in_windows = sum(len(df) for df in omt_filtered.values())
    print(f"\n🎯 Campaigns in analysis windows: {total_campaigns_in_windows}")

print(f"\n🚀 READY FOR SPEC-COMPLIANT BUNDLE ANALYSIS")
print(f"📅 Loaded at: {analysis_runtime.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"=" * 60)

🧊 SNAPSHOT DATA LOADING
📅 Analysis Runtime: 2025-09-29 16:45:51
📁 Snapshot Directory: C:\Users\Marco.Africani\OneDrive - AVU SA\AVU CPI Campaign\Puzzle_control_Reports\IRON_DATA\snapshots

⏰ Analysis Time Windows:
   7d: Campaigns after 2025-09-22 16:45
   14d: Campaigns after 2025-09-15 16:45
   30d: Campaigns after 2025-08-30 16:45

📊 Loading Snapshot Files:
   ✅ OMT Main Offer: 4,847 records × 21 columns
   ✅ Power BI Statistics: 30,516 records × 58 columns
   ✅ Stock List: 4,715 records × 42 columns
   ✅ Client Lines: 61 records × 30 columns
   ✅ Inactive Clients: 2,429 records × 29 columns

🔍 Data Validation & Column Mapping:
   📋 OMT Main Offer (4847 campaigns):
      📅 Schedule column: 'schedule datetime'
      ✅ Valid schedule dates: 4847
      📋 Campaign column: 'campaign status'
      📧 Email column: 'number of sent emails'
   📊 Power BI Statistics (30516 records):
      📋 Campaign No: 'campaign no.'
      📋 Main Campaign: 'main campaign no.'
      💰 LCY Amount: 'total bottle

In [23]:
# === SPEC-COMPLIANT WINNERS (ENHANCED: MinQty=0, Multi-source Vintage) ===
import re
import numpy as np
import pandas as pd

CONV_WEIGHT, SALES_WEIGHT = 0.6, 0.4
RESURRECTION_BONUS_PER = 0.02
RESURRECTION_CAP = 0.10

def _norm_cm(s):
    if pd.isna(s): return ""
    s = str(s).strip()
    s = re.sub(r"\.0$", "", s)
    s = re.sub(r"[^A-Za-z0-9]", "", s).upper()
    s = re.sub(r"^CM", "", s)
    return s.lstrip("0")

def _norm_item(x): 
    return str(x).strip().replace(".0","")

def _looks_75cl(val: str) -> bool:
    return bool(re.search(r"(75\s*cl|0\.75\s*l|750\s*ml|0,75\s*l)", str(val).lower()))

def _minmax(x: pd.Series) -> pd.Series:
    x = pd.to_numeric(x, errors="coerce").fillna(0.0)
    if x.empty: return x
    lo, hi = float(x.min()), float(x.max())
    if not np.isfinite(lo) or not np.isfinite(hi) or hi == lo:
        return pd.Series(np.ones(len(x)), index=x.index)
    return (x - lo) / (hi - lo)

# ---- Column picks (tolerant) ----
def _pick(cols, *names, regex=None):
    cols = [str(c).lower() for c in cols]
    # exact-ish first
    for n in names:
        if n and n.lower() in cols:
            i = cols.index(n.lower()); return i
    # regex fallback
    if regex:
        rx = re.compile(regex, re.I)
        for i, c in enumerate(cols):
            if rx.search(c): return i
    return None

# === 1) Build bundle_id on stats (HORECA excluded) ===
df = stats_df.copy()

# Identify columns
i_cm   = _pick(df.columns, "campaign no.", "campaign code", regex=r"\bcampaign\b.*(no|code)")
i_main = _pick(df.columns, "main campaign no.", "parent campaign no.", regex=r"^main.*campaign")
i_amt  = _pick(df.columns, "total bottle amount (lcy)", regex=r"total.*bottle.*amount.*lcy")
i_qty  = _pick(df.columns, "total bottle quantity", "quantity", "qty", regex=r"(bottle.*)?qty|quantity")
i_cust = _pick(df.columns, "customer no.", "contact no.", regex=r"^(customer|contact).*(no|id)")
i_item = _pick(df.columns, "item no.", "item number", regex=r"^item(\s|_)?(no|number)?$")
i_size = _pick(df.columns, "bottle size", "item size", "volume", regex=r"(bottle|item|unit|uom).*(size|vol|ml|cl|l)")
i_date = _pick(df.columns, "posting date", "document date", "sales date", "date", regex=r"(posting|document|sales).*date")
i_seg  = _pick(df.columns, "segment code", "segment", regex=r"segment.*code")
i_name = _pick(df.columns, "wine name", "item description", "description", "name", regex=r"(wine\s*name|item\s*description|description|name)")

if i_seg is not None:
    mask_exclude = df.iloc[:, i_seg].astype(str).str.strip().str.lower().isin(["horeca", "trade"])
    df = df.loc[~mask_exclude].copy()

df["cm_no"]     = df.iloc[:, i_cm].map(_norm_cm) if i_cm is not None else ""
df["main_cm"]   = df.iloc[:, i_main].map(_norm_cm) if i_main is not None else ""
df["bundle_id"] = np.where(df["main_cm"]!="", df["main_cm"], df["cm_no"])

if i_amt is not None:
    df["amount"] = pd.to_numeric(df.iloc[:, i_amt], errors="coerce").fillna(0.0)
else:
    df["amount"] = 0.0

if i_qty  is not None: df["qty"]       = pd.to_numeric(df.iloc[:, i_qty], errors="coerce")
if i_cust is not None: df["customer_no"] = df.iloc[:, i_cust].astype(str).str.strip().str.replace(r"\.0$","", regex=True)
if i_item is not None: df["item_no"]     = df.iloc[:, i_item].astype(str).str.strip().str.replace(r"\.0$","", regex=True)
if i_size is not None: df["size_raw"]    = df.iloc[:, i_size].astype(str).str.strip().str.lower()
if i_date is not None: df["line_date"]   = pd.to_datetime(df.iloc[:, i_date], errors="coerce")
name_col = df.columns[i_name] if i_name is not None else None

# === 2) CM→bundle map (for proper OMT aggregation & Lines mapping) ===
cm_to_bundle = df[["cm_no","bundle_id"]].drop_duplicates()

# === 3) OMT emails per bundle (Size=75.00, MinQty=0) ===
omt = omt_df.copy()
j_cm = _pick(omt.columns, "campaign no.", "campaign code", regex=r"(campaign).*(no|code)")
j_em = _pick(omt.columns, "number of sent emails", "sent emails", "emails sent",
             regex=r"^number.*sent.*email|^sent.*email|emails")
j_size = _pick(omt.columns, "size", "bottle size", "volume", regex=r"^(size|bottle.*size|volume)$")
j_minqty = _pick(omt.columns, "minimum quantity", "min quantity", "min qty", regex=r"(minimum|min).*(quantity|qty)")
j_vintage = _pick(omt.columns, "vintage", regex=r"^vintage$")  # Column N
assert j_cm is not None and j_em is not None, "OMT must have Campaign No. and Number of Sent Emails."

# Build OMT data with all filtering columns
omt_cols = [j_cm, j_em]
col_names = ["cm_no", "emails_sent"]

if j_size is not None:
    omt_cols.append(j_size)
    col_names.append("size")
if j_minqty is not None:
    omt_cols.append(j_minqty)
    col_names.append("min_qty")
if j_vintage is not None:
    omt_cols.append(j_vintage)
    col_names.append("vintage")

omt_use = omt.iloc[:, omt_cols].copy()
omt_use.columns = col_names

# Apply filters step by step
filter_steps = []
original_count = len(omt_use)

if "size" in omt_use.columns:
    omt_use["size"] = pd.to_numeric(omt_use["size"], errors="coerce")
    size_before = len(omt_use)
    omt_use = omt_use[omt_use["size"] == 75.00]
    filter_steps.append(f"Size=75.00: {size_before}→{len(omt_use)}")

if "min_qty" in omt_use.columns:
    omt_use["min_qty"] = pd.to_numeric(omt_use["min_qty"], errors="coerce")
    minqty_before = len(omt_use)
    omt_use = omt_use[omt_use["min_qty"] == 0]
    filter_steps.append(f"MinQty=0: {minqty_before}→{len(omt_use)}")

print(f"📧 OMT Email Filters: {' | '.join(filter_steps)} | Final: {len(omt_use)} records")

# Create vintage mapping from OMT
omt_vintage_map = pd.DataFrame(columns=["cm_no", "omt_vintage"])
if "vintage" in omt_use.columns:
    omt_vintage_map = omt_use[["cm_no", "vintage"]].copy()
    omt_vintage_map.columns = ["cm_no", "omt_vintage"]
    omt_vintage_map["cm_no"] = omt_vintage_map["cm_no"].map(_norm_cm)
    omt_vintage_map["omt_vintage"] = omt_vintage_map["omt_vintage"].astype(str).str.strip()
    omt_vintage_map = omt_vintage_map[~omt_vintage_map["omt_vintage"].isin(["", "nan", "None", "0", "NaN"])]
    print(f"🍷 OMT Vintage mapping: {len(omt_vintage_map)} campaigns with vintage info")

# Process email data
omt_use["cm_no"] = omt_use["cm_no"].map(_norm_cm)
omt_use["emails_sent"] = pd.to_numeric(omt_use["emails_sent"], errors="coerce").fillna(0).astype(int)

# Handle duplicates
omt_unique = omt_use.drop_duplicates(subset=["cm_no"], keep="first")
print(f"📧 OMT Duplicates handled: {len(omt_use)} → {len(omt_unique)} unique campaigns")

emails_map = omt_unique.merge(cm_to_bundle, on="cm_no", how="inner")
emails_by_bundle = (emails_map.groupby("bundle_id")["emails_sent"].sum()
                    .rename("emails_total").reset_index())

# === 4) Inactive recipients per bundle (Lines ∩ inactive2025) ===
inactive_count = pd.DataFrame(columns=["bundle_id","inactive_recipients"])
if lines_df is not None and not lines_df.empty and inactive_df is not None and not inactive_df.empty:
    # Pick columns from Lines: Campaign + Contact
    L_cm = _pick(lines_df.columns, "campaign no.", "campaign code", "campaign id", regex=r"\bcampaign\b")
    L_ct = _pick(lines_df.columns, "contact no.", "customer no.", "contact id", regex=r"^(contact|customer).*(no|id)")
    I_ct = _pick(inactive_df.columns, "contact no.", "customer no.", "contact id", regex=r"^(contact|customer).*(no|id)")
    I_cm = _pick(inactive_df.columns, "campaign no.", "campaign code", "campaign id", regex=r"\bcampaign\b")

    if L_cm is not None and L_ct is not None and I_ct is not None:
        L = lines_df.iloc[:, [L_cm, L_ct]].copy()
        L.columns = ["cm_no","contact_no"]
        L["cm_no"] = L["cm_no"].map(_norm_cm)
        L["contact_no"] = L["contact_no"].astype(str).str.strip().str.replace(r"\.0$","", regex=True)

        if I_cm is not None:
            I = inactive_df.iloc[:, [I_cm, I_ct]].copy()
            I.columns = ["cm_no","contact_no"]
            I["cm_no"] = I["cm_no"].map(_norm_cm)
            I["contact_no"] = I["contact_no"].astype(str).str.strip().str.replace(r"\.0$","", regex=True)
            LI = (L.merge(cm_to_bundle, on="cm_no", how="inner")
                    .merge(I, on=["cm_no","contact_no"], how="inner"))
        else:
            I = inactive_df.iloc[:, [I_ct]].copy()
            I.columns = ["contact_no"]
            I["contact_no"] = I["contact_no"].astype(str).str.strip().str.replace(r"\.0$","", regex=True)
            LI = (L.merge(cm_to_bundle, on="cm_no", how="inner")
                    .merge(I, on="contact_no", how="inner"))

        inactive_count = (LI.groupby("bundle_id")["contact_no"].nunique()
                            .rename("inactive_recipients").reset_index())

# === 5) Core bundle metrics ===
sales_by_bundle = (df.groupby("bundle_id")["amount"].sum()
                     .rename("sales_total").reset_index())
buyers_by_bundle = (df.dropna(subset=["bundle_id","customer_no"])
                      .drop_duplicates(["bundle_id","customer_no"])
                      .groupby("bundle_id").size()
                      .rename("buyers_count").reset_index())

bundles = (sales_by_bundle
           .merge(buyers_by_bundle, on="bundle_id", how="left")
           .merge(emails_by_bundle, on="bundle_id", how="left")
           .merge(inactive_count, on="bundle_id", how="left"))

for c in ["emails_total","buyers_count","inactive_recipients"]:
    if c in bundles.columns:
        bundles[c] = pd.to_numeric(bundles[c], errors="coerce").fillna(0).astype(int)

bundles["emails_total_effective"] = (bundles["emails_total"] - bundles["inactive_recipients"]).clip(lower=0).astype(int)
bundles["conversion_effective"] = np.where(bundles["emails_total_effective"]>0,
                                           bundles["buyers_count"]/bundles["emails_total_effective"], 0.0)

# === 6) Main item + wine name ===
item_sales = (df.groupby(["bundle_id","item_no"])["amount"].sum()
                .rename("item_sales").reset_index())
top_item = (item_sales.sort_values(["bundle_id","item_sales"], ascending=[True, False])
                     .groupby("bundle_id").head(1)
                     .rename(columns={"item_no":"main_item_no","item_sales":"main_item_sales"}))

# attach wine name
if name_col:
    tmp = df[[df.columns[i_item] if i_item is not None else "item_no", name_col]].dropna().copy()
    tmp.columns = ["item_no","wine_name"]
    tmp["item_no"] = tmp["item_no"].map(_norm_item)
    wine_map = (tmp.groupby(["item_no","wine_name"]).size()
                  .reset_index(name="n")
                  .sort_values(["item_no","n"], ascending=[True, False])
                  .drop_duplicates("item_no")[["item_no","wine_name"]])
    top_item["main_item_no"] = top_item["main_item_no"].map(_norm_item)
    top_item = top_item.merge(wine_map.rename(columns={"item_no":"main_item_no"}),
                              on="main_item_no", how="left").rename(columns={"wine_name":"main_wine_name"})

bundles = bundles.merge(top_item, on="bundle_id", how="left")

# === 7) Resurrection bonus ===
resurrect = pd.DataFrame(columns=["bundle_id","resurrection_count"])
if inactive_df is not None and not inactive_df.empty and "customer_no" in df.columns and "line_date" in df.columns:
    I_ct = _pick(inactive_df.columns, "contact no.", "customer no.", "contact id", regex=r"^(contact|customer).*(no|id)")
    if I_ct is not None:
        inactive_set = set(inactive_df.iloc[:, I_ct].astype(str).str.strip().str.replace(r"\.0$","", regex=True))
        df2025 = df.loc[(df["line_date"].notna()) & (df["line_date"] >= pd.Timestamp("2025-01-01"))].copy()
        df2025["customer_no"] = df2025["customer_no"].astype(str).str.strip().str.replace(r"\.0$","", regex=True)
        df2025["is_inactive"] = df2025["customer_no"].isin(inactive_set)
        first = (df2025.sort_values("line_date")
                        .loc[df2025["is_inactive"]]
                        .drop_duplicates(["customer_no"])
                        .loc[:, ["customer_no","bundle_id"]])
        resurrect = (first.groupby("bundle_id")["customer_no"].nunique()
                           .rename("resurrection_count").reset_index())

bundles = bundles.merge(resurrect, on="bundle_id", how="left")
bundles["resurrection_count"] = pd.to_numeric(bundles["resurrection_count"], errors="coerce").fillna(0).astype(int)
bundles["resurrection_bonus"] = (RESURRECTION_BONUS_PER * bundles["resurrection_count"]).clip(upper=RESURRECTION_CAP)

# === 8) Final score ===
bundles["sales_norm"] = _minmax(bundles["sales_total"])
bundles["conv_norm"]  = _minmax(bundles["conversion_effective"])
bundles["weighted_core"] = (CONV_WEIGHT*bundles["conv_norm"] + SALES_WEIGHT*bundles["sales_norm"]).fillna(0.0)
bundles["success_score"] = (bundles["weighted_core"] + bundles["resurrection_bonus"]).fillna(0.0)

winners = (bundles.sort_values(["success_score","sales_total"], ascending=[False, False], kind="mergesort")
                  .reset_index(drop=True))
winners.insert(0, "rank", winners.index + 1)

# === 9) OMT schedule per bundle ===
def _pick_omt_sched(df):
    for c in df.columns:
        if re.search(r"(schedule).*(date|time)", str(c), re.I): return c
    for c in df.columns:
        if re.search(r"(send|scheduled)", str(c), re.I): return c
    return None

col_sched = _pick_omt_sched(omt_df)
omt_sched = omt_df[[omt_df.columns[j_cm], col_sched]].copy() if (col_sched and j_cm is not None) else pd.DataFrame(columns=["cm_no","schedule_dt"])
if not omt_sched.empty:
    omt_sched.columns = ["cm_no","schedule_dt"]
    omt_sched["cm_no"] = omt_sched["cm_no"].map(_norm_cm)
    omt_sched["schedule_dt"] = pd.to_datetime(omt_sched["schedule_dt"], errors="coerce")
    sched_by_bundle = (omt_sched.merge(cm_to_bundle, on="cm_no", how="left")
                                 .dropna(subset=["bundle_id"])
                                 .groupby("bundle_id", as_index=False)["schedule_dt"].min())
else:
    sched_by_bundle = pd.DataFrame(columns=["bundle_id","schedule_dt"])

# === 10) FINAL TABLE WITH ENHANCED VINTAGE EXTRACTION ===

# Apply 30-day filter
today = pd.Timestamp.today().normalize()
cutoff = today - pd.Timedelta(days=30)
sched_recent = sched_by_bundle[pd.to_datetime(sched_by_bundle["schedule_dt"], errors="coerce") >= cutoff]
w_recent = winners.merge(sched_recent, on="bundle_id", how="inner")

print(f"📊 Found {len(w_recent)} campaigns scheduled within last 30 days")

# Get top 25
top25 = w_recent.head(25).copy()

# CM formatting
def _fmt_cm(bid):
    s = str(bid).strip()
    if s and s.lower() != "nan":
        if len(s) >= 7:  # e.g., 2502233 -> CM-25-02233
            return f"CM-{s[:2]}-{s[2:]}"
        elif len(s) >= 5:  # e.g., 02233 -> CM-25-02233
            return f"CM-25-{s}"
        else:
            return f"CM-25-{s.zfill(5)}"
    return None

top25["Starting Date"] = pd.to_datetime(top25["schedule_dt"], errors="coerce").dt.date
top25["CM"] = top25["bundle_id"].apply(_fmt_cm)

# Recalculate metrics
sales_recalc = df.groupby("bundle_id")["amount"].sum().rename("sales_total_fresh").reset_index()
buyers_recalc = (df.dropna(subset=["bundle_id","customer_no"])
                .drop_duplicates(["bundle_id","customer_no"])
                .groupby("bundle_id").size()
                .rename("buyers_count_fresh").reset_index())
emails_recalc = emails_by_bundle.copy().rename(columns={"emails_total": "emails_total_fresh"})

top25 = (top25
         .merge(sales_recalc, on="bundle_id", how="left")
         .merge(buyers_recalc, on="bundle_id", how="left")
         .merge(emails_recalc, on="bundle_id", how="left"))

# Fill missing values
for col in ["sales_total_fresh", "buyers_count_fresh", "emails_total_fresh"]:
    if col in top25.columns:
        top25[col] = pd.to_numeric(top25[col], errors="coerce").fillna(0)

# ENHANCED VINTAGE EXTRACTION from multiple sources
def get_vintage_from_sources(bundle_id, wine_name):
    """Extract vintage from multiple data sources with priority order"""
    vintage = ""
    source = ""
    
    # Priority 1: OMT Vintage column
    if len(omt_vintage_map) > 0:
        omt_match = omt_vintage_map[omt_vintage_map["cm_no"] == bundle_id]
        if not omt_match.empty:
            vintage = str(omt_match.iloc[0]["omt_vintage"]).strip()
            if vintage and vintage != "nan":
                source = "OMT"
                return vintage, source
    
    # Priority 2: Power BI Vintage column
    vintage_cols = [col for col in df.columns if 'vintage' in col.lower()]
    if vintage_cols:
        vintage_col = vintage_cols[0]
        powerbi_match = df[df["bundle_id"] == bundle_id][vintage_col].dropna()
        if not powerbi_match.empty:
            vintage = str(powerbi_match.iloc[0]).strip()
            if vintage and vintage not in ["", "nan", "None", "0"]:
                source = "PowerBI"
                return vintage, source
    
    # Priority 3: Extract from wine name
    if pd.notna(wine_name) and wine_name != "":
        wine_str = str(wine_name).strip()
        vintage_patterns = [
            r'\b(19[0-9]{2}|20[0-9]{2})\b',  # 1900-2099
            r"'([0-9]{2})\b",  # '95, '03 etc
            r'\b([0-9]{2})\s*$',  # ending with 2 digits
        ]
        
        for pattern in vintage_patterns:
            vintage_match = re.search(pattern, wine_str)
            if vintage_match:
                if "'" in pattern:  # Handle abbreviated years
                    year_short = vintage_match.group(1)
                    if int(year_short) <= 30:
                        vintage = f"20{year_short}"
                    else:
                        vintage = f"19{year_short}"
                else:
                    vintage = vintage_match.group(1)
                source = "WineName"
                break
    
    return vintage, source

# Apply enhanced vintage extraction
vintage_results = []
vintage_sources = []
wine_names_clean = []

for idx, row in top25.iterrows():
    bundle_id = row["bundle_id"]
    wine_name = row.get("main_wine_name", "")
    
    vintage, source = get_vintage_from_sources(bundle_id, wine_name)
    vintage_results.append(vintage)
    vintage_sources.append(source)
    
    # Clean wine name (remove vintage if extracted from name)
    if pd.notna(wine_name) and wine_name != "":
        wine_clean = str(wine_name).strip()
        if source == "WineName" and vintage:
            # Remove the vintage from wine name
            wine_clean = re.sub(rf'\b{vintage}\b', '', wine_clean).strip()
            wine_clean = re.sub(r'\s+', ' ', wine_clean).strip("- ")
        
        if len(wine_clean) > 30:
            wine_clean = wine_clean[:27] + "..."
    else:
        wine_clean = ""
    
    wine_names_clean.append(wine_clean)

top25["vintage"] = vintage_results
top25["vintage_source"] = vintage_sources  
top25["wine_name_short"] = wine_names_clean

print(f"🍷 Vintage extraction summary:")
for source in ["OMT", "PowerBI", "WineName", ""]:
    count = top25["vintage_source"].value_counts().get(source, 0)
    if source == "":
        print(f"   No vintage found: {count}")
    else:
        print(f"   From {source}: {count}")

# Create final display table
winners_top25_excel = pd.DataFrame({
    "CM":                top25["CM"],
    "Wine Name":         top25["wine_name_short"],
    "Vintage":           top25["vintage"],
    "Starting Date":     top25["Starting Date"],
    "Sales Total":       top25["sales_total_fresh"],
    "Clients":           top25["buyers_count_fresh"].astype("Int64"),
    "Total Sent":        top25["emails_total_fresh"].astype("Int64"),
    "Conversion":        pd.to_numeric(top25["conversion_effective"], errors="coerce").fillna(0.0).round(4),
    "Norm Sales":        (lambda s: ((s - s.min()) / (s.max() - s.min()) if s.max() > s.min() else s*0 + 1.0))
                         (pd.to_numeric(top25["sales_total_fresh"], errors="coerce").fillna(0.0)).round(4),
    "Norm Conv":         (lambda s: ((s - s.min()) / (s.max() - s.min()) if s.max() > s.min() else s*0 + 1.0))
                         (pd.to_numeric(top25["conversion_effective"], errors="coerce").fillna(0.0)).round(4),
    "Score":             (
        0.6 * (lambda s: ((s - s.min()) / (s.max() - s.min()) if s.max() > s.min() else s*0 + 1.0))
              (pd.to_numeric(top25["conversion_effective"], errors="coerce").fillna(0.0))
        + 0.4 * (lambda s: ((s - s.min()) / (s.max() - s.min()) if s.max() > s.min() else s*0 + 1.0))
              (pd.to_numeric(top25["sales_total_fresh"], errors="coerce").fillna(0.0))
        + (0.02 * pd.to_numeric(top25["resurrection_count"], errors="coerce").fillna(0)).clip(upper=0.10)
    ).round(6),
    "Rank":              pd.to_numeric(top25["rank"], errors="coerce").astype("Int64"),
    "Inactive":          pd.to_numeric(top25["resurrection_count"], errors="coerce").fillna(0).astype("Int64"),
})

# Sort by rank to ensure proper ordering
winners_top25_excel = winners_top25_excel.sort_values('Rank')

print(f"\n🏆 TOP {len(winners_top25_excel)} WINE CAMPAIGN WINNERS (Last 30 Days)")
print("="*100)
print(winners_top25_excel.to_string(index=False))


print(f"\n✅ Analysis Complete: {len(w_recent)} campaigns analyzed, top {len(winners_top25_excel)} shown")
print(f"📅 Schedule filter: Campaigns from {cutoff.strftime('%Y-%m-%d')} onwards")
print(f"📊 Data sources: Power BI Stats (HORECA & Trade excluded), OMT emails (Size=75.00, MinQty=0), Client Lines, Inactive 2025")

📧 OMT Email Filters: Size=75.00: 4847→2315 | MinQty=0: 2315→1516 | Final: 1516 records
🍷 OMT Vintage mapping: 1514 campaigns with vintage info
📧 OMT Duplicates handled: 1516 → 971 unique campaigns
📊 Found 104 campaigns scheduled within last 30 days
🍷 Vintage extraction summary:
   From OMT: 24
   From PowerBI: 1
   From WineName: 0
   No vintage found: 0

🏆 TOP 25 WINE CAMPAIGN WINNERS (Last 30 Days)
         CM                      Wine Name Vintage Starting Date   Sales Total  Clients  Total Sent  Conversion  Norm Sales  Norm Conv    Score  Rank  Inactive
CM-25-02259                        Masseto    2023    2025-09-02 178230.499991        8          53      0.1509      0.7373     1.0000 0.894929     8         0
CM-25-02602                          Pavie    2019    2025-09-29   8234.653682        2          22      0.0909      0.0336     0.6022 0.374731    17         0
CM-25-02474           Mazoyères-Chambertin    2021    2025-09-18  12491.670000        4          64      0.0625     